# 03 - Unsupervised baseline on BGL windows

This is a first test on the BGL 2k sample (100 windows), so the scores only show that the pipeline runs end to end.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluation.metrics import evaluate

SEED = 42
TRAIN_FRAC = 0.7
THRESHOLD_PCTL = 95

data = pd.read_csv(ROOT / "data" / "processed" / "bgl_2k_windows.csv", index_col="window")
X, y = data.drop(columns="label").to_numpy(dtype=float), data["label"].to_numpy()
print(f"Windows: {len(data)}, templates: {X.shape[1]}, anomaly windows: {y.sum()}")

Windows: 100, templates: 105, anomaly windows: 28


## Train / test split

The first 70% of windows (in time order) are used for training, keeping only the normal ones, and the last 30% for testing. Each model flags windows that score above the 95th percentile of its training scores.

In [2]:
n_train = int(len(data) * TRAIN_FRAC)
X_train_all, y_train_all = X[:n_train], y[:n_train]
X_test, y_test = X[n_train:], y[n_train:]
X_train = X_train_all[y_train_all == 0]

pd.DataFrame({
    "windows": [n_train, len(X_train), len(X_test)],
    "anomaly windows": [y_train_all.sum(), 0, y_test.sum()],
}, index=["first 70%", "train (normal only)", "test (last 30%)"])

,windows,anomaly windows
first 70%,70,15
train (normal only),55,0
test (last 30%),30,13


## PCA (reconstruction error)

Standardise the counts, keep 95% of the variance, and score each window by its squared reconstruction error.

In [3]:
scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=0.95, random_state=SEED).fit(scaler.transform(X_train))
print(f"PCA components kept: {pca.n_components_}")


def pca_score(X):
    Z = scaler.transform(X)
    return ((Z - pca.inverse_transform(pca.transform(Z))) ** 2).sum(axis=1)


pca_train_scores = pca_score(X_train)
pca_threshold = np.percentile(pca_train_scores, THRESHOLD_PCTL)
pca_pred = (pca_score(X_test) > pca_threshold).astype(int)

PCA components kept: 16


## Isolation Forest

Score = `-score_samples`, so higher means more anomalous (same direction as PCA).

In [4]:
iforest = IsolationForest(n_estimators=200, random_state=SEED).fit(X_train)


def iforest_score(X):
    return -iforest.score_samples(X)


if_train_scores = iforest_score(X_train)
if_threshold = np.percentile(if_train_scores, THRESHOLD_PCTL)
if_pred = (iforest_score(X_test) > if_threshold).astype(int)

## Results on the test windows

In [5]:
results = pd.DataFrame([
    {"model": "PCA", "threshold": pca_threshold, **evaluate(y_test, pca_pred)},
    {"model": "Isolation Forest", "threshold": if_threshold, **evaluate(y_test, if_pred)},
]).set_index("model")

results.style.format({
    "threshold": "{:.3f}",
    "precision": "{:.3f}",
    "recall": "{:.3f}",
    "f1": "{:.3f}",
    "false_alarm_rate": "{:.3f}",
})

,threshold,precision,recall,f1,false_alarm_rate,tp,fp,tn,fn
model,,,,,,,,,
PCA,13.501,0.545,0.923,0.686,0.588,12,10,7,1
Isolation Forest,0.544,0.000,0.000,0.000,0.000,0,0,17,13


In [6]:
pd.DataFrame({
    "true label": y_test,
    "PCA": pca_pred,
    "Isolation Forest": if_pred,
}, index=data.index[n_train:]).T

window,70,71,72,73,74,75,76,77,78,79,...,90,91,92,93,94,95,96,97,98,99
true label,1,0,0,1,1,1,1,0,0,0,...,0,0,0,0,0,0,0,1,1,1
PCA,1,1,1,1,1,1,1,0,0,0,...,1,1,1,1,1,1,1,1,1,1
Isolation Forest,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


## Why the two models behave so differently

Many test templates never appear in the normal training windows. PCA counts them as reconstruction error, while Isolation Forest can't split on a feature that is always 0 in training.

In [7]:
unseen = X_train.sum(axis=0) == 0
has_unseen = X_test[:, unseen].sum(axis=1) > 0
print(f"Templates never seen in normal training windows: {unseen.sum()} of {X.shape[1]}")
pd.crosstab(y_test, has_unseen, rownames=["true label"], colnames=["window has an unseen template"])

Templates never seen in normal training windows: 45 of 105


window has an unseen template,False,True
true label,,
0,8,9
1,0,13
